In [0]:
SELECT *
FROM retail_lakehouse.retail.bronze_retail_transactions
LIMIT 20;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Invoice) AS distinct_invoices,
    COUNT(DISTINCT StockCode) AS distinct_products
FROM retail_lakehouse.retail.bronze_retail_transactions;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    COUNT(*) AS cancelled_rows
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Invoice LIKE 'C%';
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    MIN(InvoiceDate) AS min_date,
    MAX(InvoiceDate) AS max_date,
    MIN(Price) AS min_price,
    MIN(Quantity) AS min_quantity
FROM retail_lakehouse.retail.bronze_retail_transactions;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    Invoice,
    StockCode,
    Description,
    Quantity,
    Price,
    CustomerID,
    Country,
    InvoiceDate
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Quantity < 0
ORDER BY Quantity ASC
LIMIT 20;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    Invoice,
    StockCode,
    Description,
    Quantity,
    Price,
    CustomerID,
    Country,
    InvoiceDate
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Price < 0
ORDER BY Price ASC
LIMIT 20;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    CASE
        WHEN Invoice LIKE 'C%' THEN 'CANCELLATION'
        WHEN Invoice LIKE 'A%' THEN 'ADJUSTMENT'
        WHEN Price = 0 AND CustomerID IS NULL THEN 'ZERO_PRICE_NO_CUSTOMER'
        WHEN Quantity < 0 THEN 'OTHER_NEGATIVE_QUANTITY'
        ELSE 'NORMAL'
    END AS transaction_category,
    COUNT(*) AS row_count
FROM retail_lakehouse.retail.bronze_retail_transactions
GROUP BY
    CASE
        WHEN Invoice LIKE 'C%' THEN 'CANCELLATION'
        WHEN Invoice LIKE 'A%' THEN 'ADJUSTMENT'
        WHEN Price = 0 AND CustomerID IS NULL THEN 'ZERO_PRICE_NO_CUSTOMER'
        WHEN Quantity < 0 THEN 'OTHER_NEGATIVE_QUANTITY'
        ELSE 'NORMAL'
    END
ORDER BY row_count DESC;

SELECT
    COUNT(*) AS zero_price_rows,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS zero_price_and_no_customer
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Price = 0;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    CASE
        WHEN Invoice LIKE 'C%' THEN 'CANCELLATION'
        WHEN Invoice LIKE 'A%' THEN 'ADJUSTMENT'
        WHEN Price = 0 AND CustomerID IS NULL THEN 'NON_SALE'
        WHEN Quantity < 0 THEN 'OTHER_NEGATIVE'
        ELSE 'OTHER'
    END AS category,
    COUNT(*) AS rows,
    MIN(Quantity) AS min_quantity,
    MAX(Quantity) AS max_quantity
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Quantity < 0
GROUP BY
    CASE
        WHEN Invoice LIKE 'C%' THEN 'CANCELLATION'
        WHEN Invoice LIKE 'A%' THEN 'ADJUSTMENT'
        WHEN Price = 0 AND CustomerID IS NULL THEN 'NON_SALE'
        WHEN Quantity < 0 THEN 'OTHER_NEGATIVE'
        ELSE 'OTHER'
    END;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CONCAT_WS(
            '|',
            Invoice,
            StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        )
    ) AS distinct_rows,
    COUNT(*) -
    COUNT(DISTINCT
        CONCAT_WS(
            '|',
            Invoice,
            StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        )
    ) AS duplicate_rows
FROM retail_lakehouse.retail.bronze_retail_transactions;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    CustomerID,
    Country,
    COUNT(*) AS duplicate_count
FROM retail_lakehouse.retail.bronze_retail_transactions
GROUP BY
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    CustomerID,
    Country
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN Invoice IS NULL THEN 1 ELSE 0 END) AS null_invoice,
    SUM(CASE WHEN StockCode IS NULL THEN 1 ELSE 0 END) AS null_stock_code,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description,
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS null_quantity,
    SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END) AS null_invoice_date,
    SUM(CASE WHEN Price IS NULL THEN 1 ELSE 0 END) AS null_price,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS null_country

FROM retail_lakehouse.retail.bronze_retail_transactions;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
select _source_file, 
    count(*) as row_count, 
    count(distinct _batch_id) as batch_count
from retail_lakehouse.retail.bronze_retail_transactions
group by `_source_file`
order by row_count desc;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    _source_file,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,
    COUNT(*) AS copies
FROM retail_lakehouse.retail.bronze_retail_transactions
GROUP BY
    _source_file,
    Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
HAVING COUNT(*) > 1
ORDER BY copies DESC
LIMIT 30;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT 
    Invoice,
    COUNT(*) AS line_count,
    COUNT(DISTINCT StockCode) as distinct_products,
    SUM(Quantity) as TotalQuantity,
    sum(quantity * price) as invoice_value
from retail_lakehouse.retail.bronze_retail_transactions
where invoice = '555524'
group by Invoice;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
SELECT
    Invoice,StockCode,Description,Quantity,Price,CustomerID,InvoiceDate
FROM retail_lakehouse.retail.bronze_retail_transactions
WHERE Invoice = '555524'
ORDER BY StockCode;
-- >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
WITH line_counts AS (
    SELECT
        Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,
        COUNT(*) AS copies
    FROM retail_lakehouse.retail.bronze_retail_transactions
    GROUP BY
        Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
)
SELECT
    SUM(copies) AS original_rows,
    COUNT(*) AS distinct_business_records,
    SUM(copies - 1) AS excess_rows,
    SUM(copies * Quantity * Price) AS original_value,
    SUM(Quantity * Price) AS value_if_exact_duplicates_removed,
    SUM(copies * Quantity) AS original_quantity,
    SUM(Quantity) AS quantity_if_exact_duplicates_removed
FROM line_counts;